In [59]:
import json
import random
import re
import time
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
from sklearn.model_selection import train_test_split
    

# Config

In [60]:
PROJECT_ROOT = Path("/Users/tun/Documents/2025.2/Machine learning/Project")
ARTIFACT_DIR = PROJECT_ROOT / "data" / "sbert_mpnet_processed"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

DATASET_NAME = "UniverseTBD/arxiv-abstracts-large"
SBERT_MODEL_NAME = "all-mpnet-base-v2"

CATEGORY_COUNT = 15
SAMPLES_PER_LABEL = 4000
RANDOM_SEED = 42
BATCH_SIZE = 16
MAX_SEQ_LENGTH = 384
NORMALIZE_EMBEDDINGS = True

TRAIN_RATIO = 0.70
VAL_RATIO = 0.10
TEST_RATIO = 0.20
    


# Helpers

In [61]:
def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


def get_device() -> str:
    if torch.backends.mps.is_available():
        return "mps"
    if torch.cuda.is_available():
        return "cuda"
    return "cpu"


set_seed(RANDOM_SEED)
DEVICE = get_device()
print(f"Using device: {DEVICE}")
    

Using device: mps


In [62]:
def clean_abstract(text: str) -> str:
    if not isinstance(text, str):
        return ""

    text = text.lower()
    text = re.sub(r"http\S+|www\.\S+", " ", text)
    text = re.sub(r"\S+@\S+", " ", text)
    text = re.sub(r"\$.*?\$", " ", text)
    text = re.sub(r"\\[a-z]+", " ", text)
    text = re.sub(r"\b\d+\b", " ", text)
    text = re.sub(r"[^a-z0-9\.,!\?]", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def split_category_string(categories: str) -> list[str]:
    return str(categories).strip().split()


def primary_broad_category(categories: str) -> str | None:
    parts = split_category_string(categories)
    if not parts:
        return None
    return parts[0].split(".")[0]


# 1. Load Dataset

In [63]:
print("=" * 60)
print("SECTION 1: Loading dataset")
print("=" * 60)

dataset = load_dataset(DATASET_NAME)
train_dataset = dataset["train"]

print(dataset)
print(f"Full train split rows: {len(train_dataset):,}")
    

SECTION 1: Loading dataset
DatasetDict({
    train: Dataset({
        features: ['id', 'submitter', 'authors', 'title', 'comments', 'journal-ref', 'doi', 'report-no', 'categories', 'license', 'abstract', 'versions', 'update_date', 'authors_parsed'],
        num_rows: 2292057
    })
})
Full train split rows: 2,292,057


In [64]:
print("\n" + "=" * 60)
print("SECTION 2: Primary category labels")
print("=" * 60)

records = []

for row in train_dataset:
    original_categories = row.get("categories", "")
    category_parts = split_category_string(original_categories)

    if len(category_parts) != 1:
        continue

    label = category_parts[0].split(".")[0]

    records.append(
        {
            "abstract_raw": row["abstract"],
            "abstract_clean": clean_abstract(row["abstract"]),
            "label": label,
            "original_categories": original_categories,
        }
    )

df = pd.DataFrame(records)
print(f"Prepared dataframe shape: {df.shape}")
print("Label strategy: single-category papers only -> broad primary topic")
df.head()
    



SECTION 2: Primary category labels
Prepared dataframe shape: (1245178, 4)
Label strategy: single-category papers only -> broad primary topic


,abstract_raw,abstract_clean,label,original_categories
0,A fully differential calculation in perturba...,a fully differential calculation in perturbati...,hep-ph,hep-ph
1,The evolution of Earth-Moon system is descri...,the evolution of earth moon system is describe...,physics,physics.gen-ph
2,We show that a determinant of Stirling cycle...,we show that a determinant of stirling cycle n...,math,math.CO
3,We study the two-particle wave function of p...,we study the two particle wave function of pai...,cond-mat,cond-mat.mes-hall
4,A rather non-standard quantum representation...,a rather non standard quantum representation o...,gr-qc,gr-qc


In [65]:
print("\n" + "=" * 60)
print("SECTION 3: Select top 15 primary topics")
print("=" * 60)

label_counts = df["label"].value_counts()
selected_labels = label_counts.head(CATEGORY_COUNT).index.tolist()

print("Selected labels:")
print(selected_labels)
print(label_counts.loc[selected_labels])
    


SECTION 3: Select top 15 primary topics
Selected labels:
['math', 'astro-ph', 'cs', 'cond-mat', 'physics', 'hep-ph', 'quant-ph', 'hep-th', 'gr-qc', 'nucl-th', 'stat', 'hep-ex', 'q-bio', 'hep-lat', 'eess']
label
math        264800
astro-ph    204267
cs          190329
cond-mat    167272
physics      81840
hep-ph       79303
quant-ph     64677
hep-th       57406
gr-qc        29167
nucl-th      18899
stat         17987
hep-ex       17133
q-bio        11131
hep-lat       9959
eess          9077
Name: count, dtype: int64


# 2. Balanced Sampling

In [66]:
print("\n" + "=" * 60)
print("SECTION 4: Balanced sampling")
print("=" * 60)

sampled_parts = []
shortages = {}

for label in selected_labels:
    label_df = df[df["label"] == label]
    sample_size = min(SAMPLES_PER_LABEL, len(label_df))

    if sample_size < SAMPLES_PER_LABEL:
        shortages[label] = int(sample_size)

    sampled_parts.append(
        label_df.sample(n=sample_size, random_state=RANDOM_SEED)
    )

sampled_df = pd.concat(sampled_parts, ignore_index=True)
sampled_df = sampled_df.sample(frac=1, random_state=RANDOM_SEED).reset_index(drop=True)

if shortages:
    print("Labels with fewer than requested samples:")
    print(shortages)

print(f"Sampled dataset shape: {sampled_df.shape}")
print("Samples per label:")
print(sampled_df["label"].value_counts().sort_index())
    


SECTION 4: Balanced sampling
Sampled dataset shape: (60000, 4)
Samples per label:
label
astro-ph    4000
cond-mat    4000
cs          4000
eess        4000
gr-qc       4000
hep-ex      4000
hep-lat     4000
hep-ph      4000
hep-th      4000
math        4000
nucl-th     4000
physics     4000
q-bio       4000
quant-ph    4000
stat        4000
Name: count, dtype: int64


In [67]:
print("\n" + "=" * 60)
print("SECTION 5: Label encoding")
print("=" * 60)

label_to_id = {label: idx for idx, label in enumerate(sorted(selected_labels))}
id_to_label = {idx: label for label, idx in label_to_id.items()}

sampled_df["label_id"] = sampled_df["label"].map(label_to_id)

print(f"Number of classes: {len(label_to_id)}")
print("Label mapping:")
print(label_to_id)
sampled_df[["abstract_clean", "label", "label_id"]].head()
    


SECTION 5: Label encoding
Number of classes: 15
Label mapping:
{'astro-ph': 0, 'cond-mat': 1, 'cs': 2, 'eess': 3, 'gr-qc': 4, 'hep-ex': 5, 'hep-lat': 6, 'hep-ph': 7, 'hep-th': 8, 'math': 9, 'nucl-th': 10, 'physics': 11, 'q-bio': 12, 'quant-ph': 13, 'stat': 14}


,abstract_clean,label,label_id
0,we show that the physical subspace in the z2 s...,cond-mat,1
1,the triple alpha reaction is a key to c produc...,nucl-th,10
2,two particle correlations based on the interfe...,nucl-th,10
3,"in this paper, we present and analyze the prop...",cs,2
4,the task of word level quality estimation qe c...,cs,2


# 3. Stratified Train/Validation/Test Split

In [68]:
print("\n" + "=" * 60)
print("SECTION 6: Stratified train/validation/test split")
print("=" * 60)
print(f"Target split ratio: train={TRAIN_RATIO:.2f}, val={VAL_RATIO:.2f}, test={TEST_RATIO:.2f}")

train_val_df, test_df = train_test_split(
    sampled_df,
    test_size=TEST_RATIO,
    stratify=sampled_df["label_id"],
    random_state=RANDOM_SEED,
)

relative_val_ratio = VAL_RATIO / (TRAIN_RATIO + VAL_RATIO)
train_df, val_df = train_test_split(
    train_val_df,
    test_size=relative_val_ratio,
    stratify=train_val_df["label_id"],
    random_state=RANDOM_SEED,
)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print(f"Train shape: {train_df.shape}")
print(f"Validation shape: {val_df.shape}")
print(f"Test shape: {test_df.shape}")
print("Actual split sizes:")
print({
    "train": len(train_df),
    "val": len(val_df),
    "test": len(test_df),
})
    


SECTION 6: Stratified train/validation/test split
Target split ratio: train=0.70, val=0.10, test=0.20
Train shape: (41999, 5)
Validation shape: (6001, 5)
Test shape: (12000, 5)
Actual split sizes:
{'train': 41999, 'val': 6001, 'test': 12000}


In [69]:
for split_name, split_df in {
    "train": train_df,
    "val": val_df,
    "test": test_df,
}.items():
    print(f"\n{split_name}")
    print(split_df["label"].value_counts().sort_index())
    


train
label
astro-ph    2800
cond-mat    2800
cs          2800
eess        2800
gr-qc       2800
hep-ex      2800
hep-lat     2799
hep-ph      2800
hep-th      2800
math        2800
nucl-th     2800
physics     2800
q-bio       2800
quant-ph    2800
stat        2800
Name: count, dtype: int64

val
label
astro-ph    400
cond-mat    400
cs          400
eess        400
gr-qc       400
hep-ex      400
hep-lat     401
hep-ph      400
hep-th      400
math        400
nucl-th     400
physics     400
q-bio       400
quant-ph    400
stat        400
Name: count, dtype: int64

test
label
astro-ph    800
cond-mat    800
cs          800
eess        800
gr-qc       800
hep-ex      800
hep-lat     800
hep-ph      800
hep-th      800
math        800
nucl-th     800
physics     800
q-bio       800
quant-ph    800
stat        800
Name: count, dtype: int64


# 4. SBERT Text Representation

In [70]:
def encode_texts(model: SentenceTransformer, texts: list[str]) -> np.ndarray:
    embeddings = model.encode(
        texts,
        batch_size=BATCH_SIZE,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=NORMALIZE_EMBEDDINGS,
    )
    return embeddings.astype(np.float32)
    

In [71]:
model = SentenceTransformer(SBERT_MODEL_NAME, device=DEVICE)
model.max_seq_length = MAX_SEQ_LENGTH

print(model)
print(f"Max sequence length: {model.max_seq_length}")
    


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

SentenceTransformer(
  (0): Transformer({'transformer_task': 'feature-extraction', 'modality_config': {'text': {'method': 'forward', 'method_output_name': 'last_hidden_state'}}, 'module_output_name': 'token_embeddings', 'architecture': 'MPNetModel'})
  (1): Pooling({'embedding_dimension': 768, 'pooling_mode': 'mean', 'include_prompt': True})
  (2): Normalize({})
)
Max sequence length: 384


In [72]:
print("\n" + "=" * 60)
print("SECTION 7: SBERT embedding")
print("=" * 60)
print(f"Embedding model: {SBERT_MODEL_NAME}")
print(f"Normalize embeddings: {NORMALIZE_EMBEDDINGS}")

embedding_start = time.perf_counter()

X_train = encode_texts(model, train_df["abstract_clean"].tolist())
X_val = encode_texts(model, val_df["abstract_clean"].tolist())
X_test = encode_texts(model, test_df["abstract_clean"].tolist())

y_train = train_df["label_id"].to_numpy(dtype=np.int64)
y_val = val_df["label_id"].to_numpy(dtype=np.int64)
y_test = test_df["label_id"].to_numpy(dtype=np.int64)

embedding_time_seconds = time.perf_counter() - embedding_start
print(f"SBERT embedding time: {embedding_time_seconds:.2f} seconds")
    


SECTION 7: SBERT embedding
Embedding model: all-mpnet-base-v2
Normalize embeddings: True


Batches:   0%|          | 0/2625 [00:00<?, ?it/s]

Batches:   0%|          | 0/376 [00:00<?, ?it/s]

Batches:   0%|          | 0/750 [00:00<?, ?it/s]

SBERT embedding time: 689.96 seconds


In [73]:
print("\nEmbedding output shapes:")
print(f"X_train: {X_train.shape}, dtype={X_train.dtype}")
print(f"X_val  : {X_val.shape}, dtype={X_val.dtype}")
print(f"X_test : {X_test.shape}, dtype={X_test.dtype}")
print(f"y_train: {y_train.shape}, dtype={y_train.dtype}")
print(f"y_val  : {y_val.shape}, dtype={y_val.dtype}")
print(f"y_test : {y_test.shape}, dtype={y_test.dtype}")
    


Embedding output shapes:
X_train: (41999, 768), dtype=float32
X_val  : (6001, 768), dtype=float32
X_test : (12000, 768), dtype=float32
y_train: (41999,), dtype=int64
y_val  : (6001,), dtype=int64
y_test : (12000,), dtype=int64


# 5. Save Artifacts

In [74]:
embeddings_path = ARTIFACT_DIR / "sbert_mpnet_base_v2_embeddings.npz"

if embeddings_path.exists():
    embeddings_path.unlink()

np.savez_compressed(
    embeddings_path,
    X_train=X_train,
    X_val=X_val,
    X_test=X_test,
    y_train=y_train,
    y_val=y_val,
    y_test=y_test,
)

print("\n" + "=" * 60)
print("SECTION 8: Saving artifacts")
print("=" * 60)
print(f"Saved embeddings to: {embeddings_path}")
    



SECTION 8: Saving artifacts
Saved embeddings to: /Users/tun/Documents/2025.2/Machine learning/Project/data/sbert_mpnet_processed/sbert_mpnet_base_v2_embeddings.npz


In [75]:
metadata_columns = [
    "abstract_raw",
    "abstract_clean",
    "label",
    "label_id",
    "original_categories",
]

train_df[metadata_columns].to_csv(ARTIFACT_DIR / "metadata_train.csv", index=False)
val_df[metadata_columns].to_csv(ARTIFACT_DIR / "metadata_val.csv", index=False)
test_df[metadata_columns].to_csv(ARTIFACT_DIR / "metadata_test.csv", index=False)

print(f"Saved metadata files to: {ARTIFACT_DIR}")
    

Saved metadata files to: /Users/tun/Documents/2025.2/Machine learning/Project/data/sbert_mpnet_processed


In [76]:
config = {
    "dataset_name": DATASET_NAME,
    "sbert_model": SBERT_MODEL_NAME,
    "embedding_dim": int(X_train.shape[1]),
    "selected_labels": selected_labels,
    "label_to_id": label_to_id,
    "id_to_label": {str(key): value for key, value in id_to_label.items()},
    "samples_per_label": SAMPLES_PER_LABEL,
    "actual_samples_per_label": {
        label: int(count)
        for label, count in sampled_df["label"].value_counts().items()
    },
    "shortages": shortages,
    "split_ratio": {"train": TRAIN_RATIO, "val": VAL_RATIO, "test": TEST_RATIO},
    "split_sizes": {
        "train": int(len(train_df)),
        "val": int(len(val_df)),
        "test": int(len(test_df)),
    },
    "random_seed": RANDOM_SEED,
    "batch_size": BATCH_SIZE,
    "max_seq_length": MAX_SEQ_LENGTH,
    "normalize_embeddings": NORMALIZE_EMBEDDINGS,
    "embedding_dtype": "float32",
    "preprocessing": "first_try_style_cleaning",
    "label_strategy": "single_category_only_then_broad_category",
    "device": DEVICE,
    "hardware": "MacBook Pro M4 Pro, 24GB RAM, 512GB SSD",
    "embedding_time_seconds": float(embedding_time_seconds),
}

config_path = ARTIFACT_DIR / "sbert_config.json"
config_path.write_text(json.dumps(config, indent=2, ensure_ascii=False), encoding="utf-8")

print(f"Saved config to: {config_path}")
    


Saved config to: /Users/tun/Documents/2025.2/Machine learning/Project/data/sbert_mpnet_processed/sbert_config.json


# Final Check

In [77]:
print("Artifact folder:", ARTIFACT_DIR)
print("Files:")
for path in sorted(ARTIFACT_DIR.glob("*")):
    print("-", path.name)
    

Artifact folder: /Users/tun/Documents/2025.2/Machine learning/Project/data/sbert_mpnet_processed
Files:
- metadata_test.csv
- metadata_train.csv
- metadata_val.csv
- models
- sbert_config.json
- sbert_model_metrics.csv
- sbert_model_metrics.json
- sbert_mpnet_base_v2_embeddings.npz
- sbert_validation_metrics.csv
